# 🚀 Model Quantization Techniques Comparison

이 노트북에서는 **동일한 모델(TinyLlama-1.1B)**을 다양한 양자화 기법으로 비교합니다:
- **BitsAndBytes** (8-bit, 4-bit) - Fine-tuning(QLoRA)에서 가장 대중적
- **GGUF** (llama.cpp) - 로컬 추론에서 가장 대중적
- **Dynamic Quantization** (PyTorch native) - 추가 라이브러리 불필요

### 사전 설치
```bash
pip install transformers accelerate bitsandbytes llama-cpp-python huggingface_hub matplotlib pandas
```


In [ ]:
import torch
import gc
import time
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import hf_hub_download
import warnings
warnings.filterwarnings('ignore')

print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## 🔧 유틸리티 함수


In [ ]:
def get_model_size(model):
    """모델 크기를 MB 단위로 계산"""
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / 1024 / 1024

def get_gpu_memory():
    """현재 GPU 메모리 사용량"""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024**2
    return 0

def cleanup():
    """메모리 정리"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

test_prompt = "The meaning of life is"

def test_inference(model, tokenizer):
    """추론 테스트 (transformers 모델용)"""
    inputs = tokenizer(test_prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=50, do_sample=False)
    elapsed = time.time() - start_time

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return {"text": text, "time": elapsed}

def test_inference_gguf(llm):
    """추론 테스트 (GGUF 모델용)"""
    start_time = time.time()
    output = llm(test_prompt, max_tokens=50, echo=False)
    elapsed = time.time() - start_time

    text = test_prompt + output["choices"][0]["text"]
    return {"text": text, "time": elapsed}

## 🎯 테스트 모델: TinyLlama-1.1B

모든 양자화 기법을 **동일한 모델**로 비교합니다.


In [ ]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print(f"Selected Model: {model_id}")
print(f"Parameters: ~1.1B")

tokenizer = AutoTokenizer.from_pretrained(model_id)

## 📊 1. 기준 모델 (FP16)


In [ ]:
print("📊 Loading Base Model (FP16)...")
cleanup()
torch.cuda.reset_peak_memory_stats()

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map={"": 0}
)

base_size = get_model_size(base_model)
base_memory = get_gpu_memory()
base_result = test_inference(base_model, tokenizer)

print(f"✅ Base Model (FP16):")
print(f"   - Model Size: {base_size:.2f} MB")
print(f"   - GPU Memory: {base_memory:.2f} MB")
print(f"   - Inference Time: {base_result['time']:.3f} seconds")
print(f"   - Output: {base_result['text'][:120]}")

del base_model
cleanup()

## 🔵 2. BitsAndBytes 8-bit 양자화


In [ ]:
print("🔵 Loading BitsAndBytes 8-bit Model...")
cleanup()

bnb_config_8bit = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.float16
)

model_8bit = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config_8bit,
    device_map={"": 0}
)

bnb8_size = get_model_size(model_8bit)
bnb8_memory = get_gpu_memory()
bnb8_result = test_inference(model_8bit, tokenizer)

print(f"✅ BitsAndBytes 8-bit:")
print(f"   - Model Size: {bnb8_size:.2f} MB")
print(f"   - GPU Memory: {bnb8_memory:.2f} MB")
print(f"   - Size Reduction: {(1 - bnb8_memory/base_memory)*100:.1f}%")
print(f"   - Inference Time: {bnb8_result['time']:.3f} seconds")
print(f"   - Output: {bnb8_result['text'][:120]}")

del model_8bit
cleanup()

## 🔵 3. BitsAndBytes 4-bit 양자화 (NF4)


In [ ]:
print("🔵 Loading BitsAndBytes 4-bit Model (NF4)...")
cleanup()

bnb_config_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model_4bit = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config_4bit,
    device_map={"": 0}
)

bnb4_size = get_model_size(model_4bit)
bnb4_memory = get_gpu_memory()
bnb4_result = test_inference(model_4bit, tokenizer)

print(f"✅ BitsAndBytes 4-bit (NF4):")
print(f"   - Model Size: {bnb4_size:.2f} MB")
print(f"   - GPU Memory: {bnb4_memory:.2f} MB")
print(f"   - Size Reduction: {(1 - bnb4_memory/base_memory)*100:.1f}%")
print(f"   - Inference Time: {bnb4_result['time']:.3f} seconds")
print(f"   - Output: {bnb4_result['text'][:120]}")

del model_4bit
cleanup()

## 🟢 4. GGUF 양자화 (llama.cpp)

GGUF는 llama.cpp에서 사용하는 양자화 포맷입니다.
Hugging Face에서 사전 양자화된 동일 모델(TinyLlama-1.1B)을 다운로드합니다.

| 타입 | 비트 | 설명 |
|------|------|------|
| Q4_K_M | 4-bit | 크기/품질 균형 (가장 인기) |
| Q8_0 | 8-bit | 거의 원본 수준 |


In [ ]:
from llama_cpp import Llama

print("🟢 GGUF 양자화 모델 테스트")
print("=" * 50)

gguf_repo = "TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF"

gguf_variants = [
    ("tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf", "GGUF Q4_K_M"),
    ("tinyllama-1.1b-chat-v1.0.Q8_0.gguf", "GGUF Q8_0"),
]

gguf_results = {}

for gguf_file, label in gguf_variants:
    print(f"\n🔽 Downloading {label}...")
    try:
        model_path = hf_hub_download(repo_id=gguf_repo, filename=gguf_file)
        file_size_mb = os.path.getsize(model_path) / (1024 * 1024)

        print(f"   Loading {label}...")
        llm = Llama(
            model_path=model_path,
            n_ctx=512,
            n_gpu_layers=-1,
            verbose=False
        )

        result = test_inference_gguf(llm)

        print(f"✅ {label}:")
        print(f"   - File Size: {file_size_mb:.2f} MB")
        print(f"   - vs FP16: {(1 - file_size_mb/base_size)*100:.1f}% 감소")
        print(f"   - Inference Time: {result['time']:.3f} seconds")
        print(f"   - Output: {result['text'][:120]}")

        gguf_results[label] = {
            "size": file_size_mb,
            "time": result["time"],
            "text": result["text"]
        }

        del llm
        cleanup()

    except Exception as e:
        print(f"⚠️ {label} 실패: {e}")
        gguf_results[label] = {"size": 0, "time": 0, "text": ""}

## 🔴 5. Dynamic Quantization (PyTorch Native)

PyTorch 내장 양자화로 추가 라이브러리가 필요 없지만, CPU에서만 동작합니다.


In [ ]:
print("🔴 Applying Dynamic Quantization (CPU)...")
cleanup()

base_model_cpu = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float32
).to('cpu')

base_size_fp32 = get_model_size(base_model_cpu)

quantized_model = torch.quantization.quantize_dynamic(
    base_model_cpu,
    {torch.nn.Linear},
    dtype=torch.qint8
)

dynamic_size = get_model_size(quantized_model)

inputs = tokenizer(test_prompt, return_tensors="pt")
start_time = time.time()
with torch.no_grad():
    outputs = quantized_model.generate(**inputs, max_new_tokens=50, do_sample=False)
dynamic_time = time.time() - start_time
dynamic_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"✅ Dynamic Quantization (CPU, INT8):")
print(f"   - Model Size: {dynamic_size:.2f} MB")
print(f"   - vs FP32: {(1 - dynamic_size/base_size_fp32)*100:.1f}% 감소")
print(f"   - Inference Time: {dynamic_time:.3f} seconds")
print(f"   - Output: {dynamic_text[:120]}")
print(f"   - Note: CPU 전용, GPU 사용 불가")

del base_model_cpu, quantized_model
cleanup()

## 📈 6. 결과 비교


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 결과 정리
rows = [
    {"Method": "FP16 (기준)", "Size (MB)": round(base_size, 1), "GPU Mem (MB)": round(base_memory, 1),
     "Time (s)": round(base_result['time'], 3), "Backend": "GPU"},
    {"Method": "BnB 8-bit", "Size (MB)": round(bnb8_size, 1), "GPU Mem (MB)": round(bnb8_memory, 1),
     "Time (s)": round(bnb8_result['time'], 3), "Backend": "GPU"},
    {"Method": "BnB 4-bit", "Size (MB)": round(bnb4_size, 1), "GPU Mem (MB)": round(bnb4_memory, 1),
     "Time (s)": round(bnb4_result['time'], 3), "Backend": "GPU"},
]

for label, data in gguf_results.items():
    if data["size"] > 0:
        rows.append({
            "Method": label, "Size (MB)": round(data["size"], 1), "GPU Mem (MB)": 0,
            "Time (s)": round(data["time"], 3), "Backend": "GPU (llama.cpp)"
        })

rows.append({
    "Method": "Dynamic (CPU)", "Size (MB)": round(dynamic_size, 1), "GPU Mem (MB)": 0,
    "Time (s)": round(dynamic_time, 3), "Backend": "CPU"
})

df = pd.DataFrame(rows)

# 크기 감소율 계산
df["Size Reduction"] = df["Size (MB)"].apply(lambda x: f"{(1 - x/base_size)*100:.1f}%")

print("\n📊 === TinyLlama-1.1B 양자화 비교 ===\n")
print(df.to_string(index=False))

In [ ]:
# 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2196F3', '#4CAF50', '#8BC34A', '#FF9800', '#FF5722', '#9E9E9E']

# File/Model Size
axes[0].barh(df['Method'], df['Size (MB)'], color=colors[:len(df)])
axes[0].set_xlabel('Size (MB)')
axes[0].set_title('Model/File Size Comparison')
axes[0].invert_yaxis()
for i, v in enumerate(df['Size (MB)']):
    axes[0].text(v + 10, i, f'{v:.0f} MB', va='center')

# Inference Time
axes[1].barh(df['Method'], df['Time (s)'], color=colors[:len(df)])
axes[1].set_xlabel('Time (seconds)')
axes[1].set_title('Inference Time Comparison')
axes[1].invert_yaxis()
for i, v in enumerate(df['Time (s)']):
    axes[1].text(v + 0.01, i, f'{v:.3f}s', va='center')

plt.tight_layout()
plt.savefig('quantization_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 차트가 quantization_comparison.png로 저장되었습니다.")

## 💡 7. 양자화 방법별 특징

| 기법 | 장점 | 단점 | 주요 용도 |
|------|------|------|-----------|
| BitsAndBytes | 설정 간단, QLoRA Fine-tuning 가능 | 추론 속도 느림 | Fine-tuning |
| GGUF (llama.cpp) | CPU/GPU 모두 지원, 다양한 레벨 | Fine-tuning 불가 | 로컬 추론, Ollama |
| Dynamic Quant | 추가 라이브러리 불필요 | CPU 전용 | 간단한 배포 |

### 🎯 실전 가이드
- **Fine-tuning이 목적** → BitsAndBytes 4-bit (QLoRA)
- **로컬 추론이 목적** → GGUF Q4_K_M
- **프로덕션 서빙** → vLLM + AWQ 또는 GGUF
- **Consumer GPU (8GB 이하)** → 4-bit 양자화 필수
- **Mid-range GPU (16GB)** → 8-bit 또는 4-bit
- **High-end GPU (24GB+)** → FP16 또는 8-bit


In [ ]:
cleanup()
print("✅ 모든 테스트 완료!")